# Project FLASH V1.2 — PYNQ-Z2 board run

Runs the V1.2 (224x224) pneumonia classifier on the FPGA and checks it
bit-exact against the golden model on the 244 exported verification images.

Folder layout (made by `v1/board/make_board_bundle.ps1`): `flash.bit`,
`flash.hwh`, this notebook, `tools/flash_preprocess.py`, `vectors/`.

Build-side FCLK0 (actual, from Vivado): **66.666672 MHz**.

Register map (`top_v1_axi`): 0x00 CTRL (bit0 start, bit1 soft reset),
0x04 STATUS (bit0 busy, bit1 done, bit2 ready_for_pixels, bit3 err),
0x08 THRESHOLD, 0x0C LOGIT0, 0x10 LOGIT1, 0x14 MARGIN, 0x18 DECISION,
0x1C VERSION, 0x20 CYCLES.

In [ ]:
from pynq import Overlay, allocate, Clocks
import numpy as np, time

FCLK_BUILD_MHZ = 66.666672          # Vivado's actual FCLK0 for this bitstream

ol  = Overlay('flash.bit')
print('IP blocks:', list(ol.ip_dict.keys()))
dma = ol.axi_dma_0
acc = ol.top_v1_axi_0

version = acc.read(0x1C)
assert version == 0xF1A50102, f'wrong bitstream: VERSION = {version:#010x}'
print(f'VERSION OK ({version:#010x})')

fclk = Clocks.fclk0_mhz
print(f'FCLK0 now: {fclk:.6f} MHz (build: {FCLK_BUILD_MHZ} MHz)')
if abs(fclk - FCLK_BUILD_MHZ) > 0.01:
    print('WARNING: FCLK0 differs from the build value; timing was closed at the build value')

In [ ]:
def s32(x):
    return x - (1 << 32) if x & 0x80000000 else x

def readmem(path, signed32=False):
    v = [int(l.split('//')[0], 16) for l in open(path)
         if l.strip() and not l.startswith(('//', '@'))]
    return [s32(x) for x in v] if signed32 else v

N   = 50176                          # 224 x 224 uint8 pixels, row-major
buf = allocate(shape=(N,), dtype=np.uint8)

def run(img, thr=-646):
    """One image through the accelerator. Returns (logit0, logit1, margin, decision, cycles)."""
    buf[:] = img
    buf.flush()
    acc.write(0x08, thr & 0xFFFFFFFF)          # MMIO takes unsigned values
    acc.write(0x00, 1)                         # arm the run; pixels go in before compute starts
    dma.sendchannel.transfer(buf)
    dma.sendchannel.wait()
    while not (acc.read(0x04) & 0x2):          # STATUS.done
        pass
    assert not (acc.read(0x04) & 0x8), 'STATUS.err: stream length error'
    return (s32(acc.read(0x0C)), s32(acc.read(0x10)), s32(acc.read(0x14)),
            acc.read(0x18) & 1, acc.read(0x20))

## 244-image bit-exact sweep

In [ ]:
V  = 'vectors/'
e0 = readmem(V + 'exp_logit0.mem', True)
e1 = readmem(V + 'exp_logit1.mem', True)
em = readmem(V + 'exp_margin.mem', True)
ed = readmem(V + 'exp_decision.mem')
gt = readmem(V + 'gt_label.mem')

ok = {'logit0': 0, 'logit1': 0, 'margin': 0, 'decision': 0}
n_exact, cyc, dec, bad = 0, [], [], []
t0 = time.time()
for k in range(244):
    img = np.array(readmem(f'{V}img_{k}.mem'), dtype=np.uint8)
    l0, l1, m, d, c = run(img)
    got, exp = (l0, l1, m, d), (e0[k], e1[k], em[k], ed[k])
    for name, g, e in zip(ok, got, exp):
        ok[name] += (g == e)
    n_exact += (got == exp)
    cyc.append(c); dec.append(d)
    if got != exp:
        bad.append((k, got, exp))
        print('MISMATCH', k, got, exp)
wall = time.time() - t0

print(f'BOARD: {n_exact}/244 bit-exact')
for name, n in ok.items():
    print(f'  {name:9s} {n}/244')

## Latency

In [ ]:
cmin, cmax = min(cyc), max(cyc)
ms = lambda c: c / (fclk * 1e3)
print(f'CYCLES/image: {cmin} .. {cmax}')
print(f'compute latency @ {fclk:.6f} MHz: {ms(cmin):.2f} .. {ms(cmax):.2f} ms '
      f'-> {1000 / ms(cmax):.2f} images/s (compute only)')
print(f'wall: {wall:.1f} s total, {wall / 244 * 1000:.0f} ms/image incl. DMA and .mem parsing')

## Threshold demo — same image, two thresholds

In [ ]:
img0 = np.array(readmem(V + 'img_0.mem'), dtype=np.uint8)
for thr in (-646, 0x7FFFFFFF):
    l0, l1, m, d, c = run(img0, thr)
    print(f'image 0, threshold {thr:>11d}: margin {m}, decision {d}')

## Sensitivity / specificity vs `gt_label.mem`

Small-sample check on the 244 seeded verification images only. The headline
accuracy stays the notebook's test AUROC (V1.2: 0.8229) with its CI.

In [ ]:
tp = sum(1 for d, g in zip(dec, gt) if d == 1 and g == 1)
fn = sum(1 for d, g in zip(dec, gt) if d == 0 and g == 1)
tn = sum(1 for d, g in zip(dec, gt) if d == 0 and g == 0)
fp = sum(1 for d, g in zip(dec, gt) if d == 1 and g == 0)
print(f'TP {tp}  FN {fn}  TN {tn}  FP {fp}')
print(f'sensitivity {tp / max(tp + fn, 1):.3f}   specificity {tn / max(tn + fp, 1):.3f}   (n = 244, small sample)')

## Optional: one DICOM end to end

Needs `pydicom` and `opencv` on the board (`pip install pydicom` if missing).
Compare the logits with the golden model run on the PC for the same file.

In [ ]:
# import sys; sys.path.insert(0, 'tools')
# from flash_preprocess import preprocess_file
# arrays, info = preprocess_file('test.dcm', sizes=(224,))
# print(info)
# print(run(arrays[224].reshape(-1)))

In [ ]:
buf.freebuffer()